# Advanced Expense Categorization Pipeline with Time-Series Features

This notebook implements:
1. **Hybrid Text Features**: TF-IDF Word + Char + Gensim Doc2Vec
2. **Expanded Domain Rules & Structural Features**: Merchant rules + text metrics
3. **Time-Series Features**: Calendar, Payday/Cycle, Cyclical Sin/Cos, Date Aggregations
4. **Optuna Hyperparameter Optimization**: Tuning LightGBM, XGBoost, CatBoost
5. **OOF Evaluation & Confusion Matrix**
6. **Optimal Weighted Blending (SLSQP)**


In [ ]:
"""
1. Hybrid Text Features (TF-IDF Word + Char + Doc2Vec)
2. Expanded Merchant Rules & Text Structural Features
3. Fast Optuna Hyperparameter Optimization for LightGBM, XGBoost, CatBoost
4. Out-of-Fold (OOF) Confusion Matrix & Error Analysis
5. Optimal Weighted Blending via Scipy Optimization
"""

import re
import unicodedata
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, f1_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from scipy.sparse import hstack
from scipy.optimize import minimize

In [ ]:
# ── 1. Load Data ───────────────────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/competitions/aurora-gate-expense-categorization-challenge'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f"Loaded Train: {train.shape}, Test: {test.shape}", flush=True)

In [ ]:
# ── 2. Text Preprocessing & Cleaning ──────────────────────────────────────────
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFKC', s)
    s = s.lower()
    s = re.sub(r'[#*\-_/\\|@&%$]', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

train['desc_clean'] = train['description'].apply(clean_text)
test['desc_clean']  = test['description'].apply(clean_text)

# Text structural metrics
for df in [train, test]:
    df['char_count']  = df['desc_clean'].apply(len)
    df['word_count']  = df['desc_clean'].apply(lambda x: len(x.split()))
    df['digit_count'] = df['description'].apply(lambda s: sum(c.isdigit() for c in str(s)))

In [ ]:
# ── 3. Expanded Domain Rules & Time-Series Features ───────────────────────────
MERCHANT_RULES = [
    # Food & Dining
    ('uber eats', 'Food & Dining'), ('doordash', 'Food & Dining'), ('grubhub', 'Food & Dining'),
    ('seamless', 'Food & Dining'), ('postmates', 'Food & Dining'), ('mcdonald', 'Food & Dining'),
    ('starbucks', 'Food & Dining'), ('chipotle', 'Food & Dining'), ('subway', 'Food & Dining'),
    ('burger king', 'Food & Dining'), ('domino', 'Food & Dining'), ('taco bell', 'Food & Dining'),
    # Transportation
    ('lyft', 'Transportation'), ('uber', 'Transportation'), ('shell', 'Transportation'),
    ('chevron', 'Transportation'), ('exxon', 'Transportation'), ('mobil', 'Transportation'),
    ('bp ', 'Transportation'), ('hertz', 'Transportation'), ('avis', 'Transportation'),
    # Subscriptions
    ('netflix', 'Subscriptions'), ('spotify', 'Subscriptions'), ('hulu', 'Subscriptions'),
    ('amazon prime', 'Subscriptions'), ('apple.com bill', 'Subscriptions'), ('disney', 'Subscriptions'),
    ('hbo', 'Subscriptions'), ('youtube', 'Subscriptions'),
    # Groceries
    ('whole foods', 'Groceries'), ('instacart', 'Groceries'), ('trader joe', 'Groceries'),
    ('kroger', 'Groceries'), ('safeway', 'Groceries'), ('aldi', 'Groceries'),
    # Health & Fitness
    ('planet fitness', 'Health & Fitness'), ('cvs pharmacy', 'Health & Fitness'),
    ('walgreens', 'Health & Fitness'), ('rite aid', 'Health & Fitness'), ('equinox', 'Health & Fitness'),
    # Bills & Utilities
    ('at&t', 'Bills & Utilities'), ('verizon', 'Bills & Utilities'), ('comcast', 'Bills & Utilities'),
    ('con edison', 'Bills & Utilities'), ('t-mobile', 'Bills & Utilities'), ('spectrum', 'Bills & Utilities'),
    # Travel
    ('delta', 'Travel'), ('united', 'Travel'), ('american air', 'Travel'),
    ('southwest', 'Travel'), ('airbnb', 'Travel'), ('expedia', 'Travel'), ('marriott', 'Travel'),
    # Shopping
    ('walmart', 'Shopping'), ('target', 'Shopping'), ('best buy', 'Shopping'),
    ('home depot', 'Shopping'), ('lowe', 'Shopping'), ('amazon', 'Shopping')
]
RULE_CATS = sorted(set(c for _, c in MERCHANT_RULES))
RULE_IDX  = {c: i + 1 for i, c in enumerate(RULE_CATS)}

def rule_feature(desc: str) -> int:
    for kw, cat in MERCHANT_RULES:
        if kw in desc:
            return RULE_IDX[cat]
    return 0

for df in [train, test]:
    df['rule_feat']  = df['desc_clean'].apply(rule_feature)
    df['log_amount'] = np.log1p(df['amount'])

def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    dt = pd.to_datetime(df['date'])
    df['year'] = dt.dt.year
    df['month'] = dt.dt.month
    df['day'] = dt.dt.day
    df['quarter'] = dt.dt.quarter
    df['dayofweek'] = dt.dt.dayofweek
    df['dayofyear'] = dt.dt.dayofyear
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = dt.dt.is_month_start.astype(int)
    df['is_month_end'] = dt.dt.is_month_end.astype(int)
    
    # Payday / Billing cycles
    df['is_payday_approx'] = df['day'].isin([1, 2, 14, 15, 16, 28, 29, 30, 31]).astype(int)
    df['days_from_month_start'] = df['day'] - 1
    days_in_month = dt.dt.days_in_month
    df['days_to_month_end'] = days_in_month - df['day']
    df['week_of_month'] = (df['day'] - 1) // 7 + 1
    
    # Cyclical sin/cos encodings
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    return df

for df in [train, test]:
    add_temporal_features(df)

# Date-based aggregations across train & test combined
all_df = pd.concat([train, test], ignore_index=True)
date_counts = all_df['date'].value_counts()
date_amount_sum = all_df.groupby('date')['amount'].sum()
date_amount_mean = all_df.groupby('date')['amount'].mean()

for df in [train, test]:
    df['date_trans_count'] = df['date'].map(date_counts).fillna(0)
    df['date_amount_sum'] = df['date'].map(date_amount_sum).fillna(0)
    df['date_amount_mean'] = df['date'].map(date_amount_mean).fillna(0)

print("Generated expanded merchant rules, temporal features, and date aggregations.", flush=True)

In [ ]:
# ── 4. Hybrid Text Features (TF-IDF + Doc2Vec) ────────────────────────────────
print("\n--- Extracting Hybrid Text Representations ---", flush=True)

# Word TF-IDF
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=800, min_df=2, sublinear_tf=True)
X_word_tr = word_vec.fit_transform(train['desc_clean'])
X_word_te = word_vec.transform(test['desc_clean'])

# Char TF-IDF
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=1200, min_df=2, sublinear_tf=True)
X_char_tr = char_vec.fit_transform(train['desc_clean'])
X_char_te = char_vec.transform(test['desc_clean'])

# Doc2Vec
all_descriptions = pd.concat([train['desc_clean'], test['desc_clean']]).reset_index(drop=True)
tagged_data = [TaggedDocument(words=text.split(), tags=[str(i)]) for i, text in enumerate(all_descriptions)]

d2v_model = Doc2Vec(vector_size=50, window=5, min_count=2, workers=4, epochs=15, seed=42)
d2v_model.build_vocab(tagged_data)
d2v_model.train(tagged_data, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

def get_d2v_vectors(df, model):
    vectors = []
    for text in df['desc_clean']:
        words = text.split()
        vectors.append(model.infer_vector(words))
    return np.array(vectors)

X_d2v_tr = get_d2v_vectors(train, d2v_model)
X_d2v_te = get_d2v_vectors(test, d2v_model)

# Scaled Numeric, Temporal & Structural Features
NUM_COLS = [
    'amount', 'log_amount', 'rule_feat', 'char_count', 'word_count', 'digit_count',
    'year', 'month', 'day', 'quarter', 'dayofweek', 'dayofyear', 'is_weekend',
    'is_month_start', 'is_month_end', 'is_payday_approx',
    'days_from_month_start', 'days_to_month_end', 'week_of_month',
    'month_sin', 'month_cos', 'day_sin', 'day_cos', 'dow_sin', 'dow_cos',
    'dayofyear_sin', 'dayofyear_cos',
    'date_trans_count', 'date_amount_sum', 'date_amount_mean'
]
scaler = StandardScaler()
X_num_tr = scaler.fit_transform(train[NUM_COLS])
X_num_te = scaler.transform(test[NUM_COLS])

# Stack all features as Sparse Matrix
X_tr_sparse = hstack([X_word_tr, X_char_tr, X_d2v_tr, X_num_tr]).tocsr()
X_te_sparse = hstack([X_word_te, X_char_te, X_d2v_te, X_num_te]).tocsr()

le = LabelEncoder()
y_train = le.fit_transform(train['category'])

print(f"Hybrid Feature Matrix shape: {X_tr_sparse.shape}", flush=True)

In [ ]:
# ── 5. Optuna Hyperparameter Tuning for Top Models ────────────────────────────
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X_tr_sparse, y_train, test_size=0.2, random_state=42, stratify=y_train)

def print_trial_callback(study, trial):
    print(f"  Trial {trial.number+1}/{len(study.trials)} finished | Val F1-Macro: {trial.value:.4f}", flush=True)

print("\n=== Running Optuna Tuning for Top Models ===", flush=True)

# 5.1 LightGBM Tuning
def objective_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 31, 63),
        'max_depth': trial.suggest_int('max_depth', 5, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.12),
        'n_estimators': trial.suggest_int('n_estimators', 150, 250, step=50),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': 4,
        'verbose': -1
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_opt_tr, y_opt_tr)
    preds = m.predict(X_opt_val)
    return f1_score(y_opt_val, preds, average='macro')

print("Tuning LightGBM...", flush=True)
study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=5, callbacks=[print_trial_callback])
print(f"  -> Best LightGBM F1-Macro: {study_lgb.best_value:.4f}", flush=True)

# 5.2 XGBoost Tuning
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 250, step=50),
        'max_depth': trial.suggest_int('max_depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.12),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'random_state': 42,
        'n_jobs': 4,
        'eval_metric': 'mlogloss'
    }
    m = xgb.XGBClassifier(**params)
    m.fit(X_opt_tr, y_opt_tr)
    preds = m.predict(X_opt_val)
    return f1_score(y_opt_val, preds, average='macro')

print("\nTuning XGBoost...", flush=True)
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=5, callbacks=[print_trial_callback])
print(f"  -> Best XGBoost F1-Macro: {study_xgb.best_value:.4f}", flush=True)

# 5.3 CatBoost Tuning
def objective_cb(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 150, 250, step=50),
        'depth': trial.suggest_int('depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.12),
        'auto_class_weights': 'Balanced',
        'verbose': 0,
        'thread_count': 4,
        'random_seed': 42
    }
    m = cb.CatBoostClassifier(**params)
    m.fit(X_opt_tr, y_opt_tr)
    preds = m.predict(X_opt_val)
    return f1_score(y_opt_val, preds, average='macro')

print("\nTuning CatBoost...", flush=True)
study_cb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_cb.optimize(objective_cb, n_trials=5, callbacks=[print_trial_callback])
print(f"  -> Best CatBoost F1-Macro: {study_cb.best_value:.4f}", flush=True)

In [ ]:
# ── 6. OOF Prediction Probability Extraction & Evaluation ─────────────────────
print("\n=== Generating 5-Fold OOF Predictions & Confusion Matrix ===", flush=True)

best_lgb_params = study_lgb.best_params.copy()
best_lgb_params.update({'class_weight': 'balanced', 'random_state': 42, 'n_jobs': 4, 'verbose': -1})

best_xgb_params = study_xgb.best_params.copy()
best_xgb_params.update({'random_state': 42, 'n_jobs': 4, 'eval_metric': 'mlogloss'})

best_cb_params = study_cb.best_params.copy()
best_cb_params.update({'auto_class_weights': 'Balanced', 'verbose': 0, 'thread_count': 4, 'random_seed': 42})

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs_lgb = np.zeros((len(train), len(le.classes_)))
oof_probs_xgb = np.zeros((len(train), len(le.classes_)))
oof_probs_cb  = np.zeros((len(train), len(le.classes_)))

test_probs_lgb = np.zeros((len(test), len(le.classes_)))
test_probs_xgb = np.zeros((len(test), len(le.classes_)))
test_probs_cb  = np.zeros((len(test), len(le.classes_)))

for tr_idx, val_idx in skf.split(X_tr_sparse, y_train):
    # LightGBM
    m_lgb = lgb.LGBMClassifier(**best_lgb_params)
    m_lgb.fit(X_tr_sparse[tr_idx], y_train[tr_idx])
    oof_probs_lgb[val_idx] = m_lgb.predict_proba(X_tr_sparse[val_idx])
    test_probs_lgb += m_lgb.predict_proba(X_te_sparse) / 5.0
    
    # XGBoost
    m_xgb = xgb.XGBClassifier(**best_xgb_params)
    m_xgb.fit(X_tr_sparse[tr_idx], y_train[tr_idx])
    oof_probs_xgb[val_idx] = m_xgb.predict_proba(X_tr_sparse[val_idx])
    test_probs_xgb += m_xgb.predict_proba(X_te_sparse) / 5.0

    # CatBoost
    m_cb = cb.CatBoostClassifier(**best_cb_params)
    m_cb.fit(X_tr_sparse[tr_idx], y_train[tr_idx])
    oof_probs_cb[val_idx] = m_cb.predict_proba(X_tr_sparse[val_idx])
    test_probs_cb += m_cb.predict_proba(X_te_sparse) / 5.0

print(f"  LGBM OOF F1-Macro: {f1_score(y_train, np.argmax(oof_probs_lgb, axis=1), average='macro'):.4f}", flush=True)
print(f"  XGB  OOF F1-Macro: {f1_score(y_train, np.argmax(oof_probs_xgb, axis=1), average='macro'):.4f}", flush=True)
print(f"  CB   OOF F1-Macro: {f1_score(y_train, np.argmax(oof_probs_cb, axis=1), average='macro'):.4f}", flush=True)

# Confusion Matrix for LightGBM
cm = confusion_matrix(y_train, np.argmax(oof_probs_lgb, axis=1))
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print("\n--- OOF Confusion Matrix (LightGBM) ---", flush=True)
print(cm_df.to_string(), flush=True)

In [ ]:
# ── 7. Optimal Weighted Blending ───────────────────────────────────────────────
print("\n=== Optimizing Ensemble Weights ===", flush=True)

def loss_func(weights):
    w1, w2, w3 = weights
    blend_oof = w1 * oof_probs_lgb + w2 * oof_probs_xgb + w3 * oof_probs_cb
    preds = np.argmax(blend_oof, axis=1)
    return -f1_score(y_train, preds, average='macro')

init_weights = [1/3, 1/3, 1/3]
bounds = [(0, 1), (0, 1), (0, 1)]
constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})

res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
w1, w2, w3 = res.x
best_f1 = -res.fun

print(f"Optimal Weights -> LightGBM: {w1:.3f}, XGBoost: {w2:.3f}, CatBoost: {w3:.3f}", flush=True)
print(f"✨ Final Weighted Ensemble OOF F1-Macro: {best_f1:.4f}", flush=True)

In [ ]:
# ── 8. Generate Final Submission File ──────────────────────────────────────────
final_test_probs = w1 * test_probs_lgb + w2 * test_probs_xgb + w3 * test_probs_cb
final_preds_idx  = np.argmax(final_test_probs, axis=1)
final_preds      = le.inverse_transform(final_preds_idx)

submission = pd.DataFrame({
    'transaction_id': test['transaction_id'],
    'category':       final_preds
})
submission.to_csv('submission.csv', index=False)
print("advanced submission to: submission.csv")

print("\nPrediction distribution:")
print(pd.Series(final_preds).value_counts())